# BERT Fine-Tuning on Kaggle IMDB (via `kagglehub`)

This Colab-ready notebook downloads the IMDB 50K movie reviews dataset **without** needing `kaggle.json` (public dataset) using **kagglehub**, fine‑tunes **`bert-base-uncased`** for binary sentiment classification, evaluates the model, and runs quick predictions.

**Outline**
1. Install dependencies
2. Imports & environment setup
3. Download dataset (kagglehub)
4. Load & preprocess data
5. Train/validation/test split (80/10/10, stratified)
6. Tokenizer, PyTorch dataset & collator
7. Model & metrics
8. TrainingArguments & Trainer
9. Train & evaluate
10. Quick predictions


In [1]:
# (1) Install dependencies
# Note: We keep the current PyTorch in Colab to avoid CUDA mismatches.
%pip -q install -U kagglehub transformers scikit-learn pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.


In [2]:
# (2) Imports & environment setup
import os, glob, inspect
import numpy as np
import pandas as pd
import kagglehub
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, set_seed
)

# Optional: disable Weights & Biases auto-logging in case the environment enables it by default
os.environ["WANDB_DISABLED"] = "true"

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


Torch: 2.8.0+cu126 | CUDA available: True


In [3]:
# (3) Download dataset with kagglehub (IMDB 50K Movie Reviews)
# This does NOT require kaggle.json for public datasets.
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

# Find the CSV (usually 'IMDB Dataset.csv')
csv_candidates = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
if not csv_candidates:
    raise FileNotFoundError("Could not find a CSV in the downloaded KaggleHub directory.")

# Prefer a CSV whose filename starts with 'imdb', otherwise take the first one
csv_path_candidates = [p for p in csv_candidates if os.path.basename(p).lower().startswith("imdb") and p.lower().endswith(".csv")]
csv_path = csv_path_candidates[0] if csv_path_candidates else csv_candidates[0]
print("Using CSV:", csv_path)


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews
Using CSV: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [4]:
# (4) Load & preprocess data (binary labels: negative=0, positive=1)
df = pd.read_csv(csv_path)
expected_cols = {"review", "sentiment"}
if not expected_cols.issubset(set(df.columns)):
    raise ValueError("Unexpected CSV format. Expected columns: 'review', 'sentiment'.")

# Basic cleaning: ensure strings and map labels
df["text"] = df["review"].astype(str)
df["label"] = (df["sentiment"].str.lower() == "positive").astype(int)
df = df[["text", "label"]]

df.head()


,text,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [5]:
# (5) Train/validation/test split (80/10/10), stratified over the whole dataset
df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["label"], random_state=42
)

# Optional: tidy indices
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Splits -> train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")


Splits -> train: 40000, val: 5000, test: 5000


In [6]:
# (6) Tokenizer, PyTorch dataset & DataCollator
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_LEN = 256  # adjust as needed (trade-off between coverage and speed/memory)

def tokenize_batch(texts):
    # We leave padding to the DataCollator to dynamically pad per batch
    return tokenizer(texts, truncation=True, padding=False, max_length=MAX_LEN)

class IMDBDataset(torch.utils.data.Dataset):
    """Simple torch Dataset returning dicts compatible with HF's DataCollator."""
    def __init__(self, texts, labels):
        enc = tokenize_batch(texts.tolist())
        self.items = []
        # Build item dicts aligned with HF Trainer expectations
        for i in range(len(labels)):
            self.items.append({
                "input_ids": enc["input_ids"][i],
                "attention_mask": enc["attention_mask"][i],
                "labels": int(labels.iloc[i]),
            })
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]

train_ds = IMDBDataset(df_train["text"], df_train["label"])
val_ds   = IMDBDataset(df_val["text"],   df_val["label"])
test_ds  = IMDBDataset(df_test["text"],  df_test["label"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
# (7) Model & metrics
# id2label/label2id are optional but recommended for clarity and portability
classes = ["negative", "positive"]
id2label = {i: c for i, c in enumerate(classes)}
label2id = {c: i for i, c in enumerate(classes)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(classes), id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# (8) TrainingArguments (compatible with multiple transformers versions) & Trainer
set_seed(42)  # reproducibility

sig_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
kwargs = dict(
    output_dir="out-bert-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,   # bump to 3–4 for better final scores
    weight_decay=0.01,
    logging_steps=50,
)

# Prefer bf16 on Ampere+ if supported; fallback to fp16 if available
bf16_supported = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
if "bf16" in sig_params and torch.cuda.is_available() and bf16_supported:
    kwargs["bf16"] = True
elif "fp16" in sig_params and torch.cuda.is_available():
    kwargs["fp16"] = True

# Evaluation/saving strategy
if "evaluation_strategy" in sig_params:
    kwargs.update(
        evaluation_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="none",
    )
    if "save_strategy" in sig_params:
        kwargs["save_strategy"] = "epoch"
        kwargs["save_total_limit"] = 1
elif "evaluate_during_training" in sig_params:
    # For older transformers versions
    kwargs.update(
        evaluate_during_training=True,
        eval_steps=500,
        logging_steps=50,
    )

args = TrainingArguments(**kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds if ("evaluation_strategy" in sig_params or "evaluate_during_training" in sig_params) else None,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-503379760.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [9]:
# (9) Train & evaluate
trainer.train()
print("VALID:", trainer.evaluate(val_ds))
print("TEST :", trainer.evaluate(test_ds))


Step,Training Loss
50,0.565400
100,0.367400
150,0.328400
200,0.302600
250,0.342200
300,0.292600
350,0.297400
400,0.279900
450,0.239700
500,0.294600


VALID: {'eval_loss': 0.257282018661499, 'eval_accuracy': 0.9268, 'eval_macro_f1': 0.9267980206184776, 'eval_runtime': 2.9578, 'eval_samples_per_second': 1690.422, 'eval_steps_per_second': 53.079, 'epoch': 2.0}
TEST : {'eval_loss': 0.2463400810956955, 'eval_accuracy': 0.9318, 'eval_macro_f1': 0.9317962651634804, 'eval_runtime': 3.0704, 'eval_samples_per_second': 1628.432, 'eval_steps_per_second': 51.133, 'epoch': 2.0}


In [10]:
# (10) Quick predictions
def predict(texts):
    batch = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(**batch).logits
    return [id2label[i] for i in logits.argmax(dim=-1).cpu().tolist()]

print(predict([
    "Absolutely fantastic movie. I loved every minute!",
    "It was okay, not great, not terrible.",
    "Terrible plot and bad acting."
]))


['positive', 'negative', 'negative']
